# Implement Workflow Patterns with LangGraph — Ollama Local

**Adapted for Ollama Local Inference**

Original course: IBM Skills Network (Kunal Makwana)  
Original LLM: OpenAI `gpt-4o-mini`  
Adapted to: Ollama local (100% offline, fully sovereign)

Estimated time needed: **45 minutes**

## Introduction

In this lab, you'll master the three essential workflow patterns that transform individual AI models into sophisticated, coordinated systems. Through hands-on projects using LangGraph and Ollama, you'll build job application assistants, intelligent task routers, and multilingual processors that demonstrate real-world multi-agent coordination.

These proven patterns - **Prompt Chaining**, **Routing**, and **Parallelization** - form the foundation of every enterprise AI system. By the end, you'll have the architectural knowledge to create AI applications that intelligently orchestrate multiple specialized agents to solve complex problems.

All inferences run **100% locally** using Ollama — no cloud API calls, no data leaving your infrastructure.

## Setup: Install Dependencies and Configure Ollama

In [ ]:
%%capture
%pip install langchain>=0.3.0
%pip install langchain-community>=0.3.0
%pip install langgraph>=0.2.0
%pip install langchain-ollama>=0.1.0
%pip install python-dotenv>=1.0.0
%pip install pydantic>=2.0.0
%pip install typing-extensions>=4.8.0

## Import Required Libraries

In [ ]:
import configparser
from langgraph.graph import StateGraph, END, START
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from langchain_ollama import OllamaLLM
from typing_extensions import TypedDict
import logging
from IPython.display import Image, display

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

logger.info("✓ Imports successful")

## Configure Ollama LLM

In [ ]:
# Load configuration from config.ini
config = configparser.ConfigParser()
config.read('config.ini')

# Initialize Ollama LLM
llm = OllamaLLM(
    model=config.get('ollama', 'model_name', fallback='qwen2.5:7b'),
    base_url=config.get('ollama', 'base_url', fallback='http://localhost:11434'),
    temperature=float(config.get('ollama', 'temperature', fallback='0.7')),
)

logger.info(f"✓ Ollama LLM initialized: {config.get('ollama', 'model_name')}")
logger.info(f"  Base URL: {config.get('ollama', 'base_url')}")
logger.info(f"  Temperature: {config.get('ollama', 'temperature')}")

## Utility Functions

In [ ]:
def print_workflow_info(workflow, app=None):
    """Prints comprehensive information about a LangGraph workflow."""
    print("WORKFLOW INFORMATION")
    print("=" * 50)
    print(f"Nodes: {list(workflow.nodes.keys())}")
    print(f"Edges: {workflow.edges}")
    
    if app:
        print("\nWorkflow Visualization generated successfully.")

---
# PATRÓN 1: PROMPT CHAINING

## Descripción

Prompt Chaining es un patrón de diseño de workflow donde tareas complejas se descomponen en una secuencia de llamadas LLM. Cada paso depende del output del anterior, permitiendo refinamiento paso a paso.

**Estructura típica:**
- Step 1: LLM prompt inicial (ej. generar borrador)
- Step 2: Prompt de refinamiento (ej. mejorar estilo)
- Step 3: Evaluación o formateo (ej. convertir a formato específico)

**Casos de uso:**
- Generar posts de blog paso a paso (idea → outline → párrafos → pulir)
- Generación automatizada de reportes (extraer → analizar → resumir)
- Creación de contenido educativo (tema → preguntas → respuestas → explicaciones)

## Use Case: Job Application Assistant

Construiremos un **asistente de solicitud laboral** usando Prompt Chaining:
1. Leer **job description** y generar **resume summary** personalizado
2. Usar ese resumen para generar **cover letter** profesional

In [ ]:
# Define the state structure for Prompt Chaining
class ChainState(TypedDict):
    job_description: str
    resume_summary: str
    cover_letter: str

logger.info("✓ ChainState defined")

In [ ]:
# Node 1: Generate Resume Summary
def generate_resume_summary(state: ChainState) -> ChainState:
    """Generates a resume summary based on job description."""
    prompt = f"""
You're a resume assistant. Read the following job description and summarize the key qualifications and experience the ideal candidate should have, phrased as if from the perspective of a strong applicant's resume summary.

Job Description:
{state['job_description']}

Provide a concise resume summary (2-3 sentences).
"""
    response = llm.invoke(prompt)
    return {**state, "resume_summary": response}

logger.info("✓ generate_resume_summary node defined")

In [ ]:
# Node 2: Generate Cover Letter
def generate_cover_letter(state: ChainState) -> ChainState:
    """Generates a cover letter using resume summary and job description."""
    prompt = f"""
You're a cover letter writing assistant. Using the resume summary below, write a professional and personalized cover letter for the following job.

Resume Summary:
{state['resume_summary']}

Job Description:
{state['job_description']}

Write a 2-3 paragraph professional cover letter.
"""
    response = llm.invoke(prompt)
    return {**state, "cover_letter": response}

logger.info("✓ generate_cover_letter node defined")

In [ ]:
# Build the Prompt Chaining workflow
workflow = StateGraph(ChainState)

# Add nodes
workflow.add_node("generate_resume_summary", generate_resume_summary)
workflow.add_node("generate_cover_letter", generate_cover_letter)

# Set entry point
workflow.set_entry_point("generate_resume_summary")

# Add edges (sequential)
workflow.add_edge("generate_resume_summary", "generate_cover_letter")
workflow.add_edge("generate_cover_letter", END)

# Compile
app_chain = workflow.compile()

print_workflow_info(workflow)
logger.info("✓ Prompt Chaining workflow compiled")

In [ ]:
# Test Prompt Chaining
input_state = {
    "job_description": "We are looking for a data scientist with experience in machine learning, NLP, and Python. Prior work with large datasets and experience deploying models into production is required.",
    "resume_summary": "",
    "cover_letter": ""
}

logger.info("Executing Prompt Chaining workflow...")
result = app_chain.invoke(input_state)

print("\n" + "="*70)
print("PROMPT CHAINING - RESULTS")
print("="*70)
print("\n📄 RESUME SUMMARY:")
print(result['resume_summary'])
print("\n💌 COVER LETTER:")
print(result['cover_letter'])

---
# PATRÓN 2: ROUTING

## Descripción

Routing es un patrón donde un LLM **clasifica o interpreta input entrante** y luego lo **enruta al proceso o agente apropiado**. Funciona como un switchboard — un nodo inteligente (clasificador o router) analiza el input y lo dirige a la rama correcta.

**Casos de uso:**
- Bots de atención al cliente con IA (enrutar billing, tech support, inquiries generales)
- Agentes multi-skill (ej. summarization, translation, data extraction)
- Bots de educación adaptativa (enrutar a módulos de math, science, grammar)

## Use Case: Task Classifier (Summarize vs Translate)

In [ ]:
# Define state for Routing
class RouterState(TypedDict):
    user_input: str
    task_type: str
    output: str

logger.info("✓ RouterState defined")

In [ ]:
# Router node: Classifies user intent
def router_node(state: RouterState) -> RouterState:
    """Classifies whether user wants to summarize or translate."""
    routing_prompt = f"""
You are an AI task classifier.

Decide whether the user wants to:
- "summarize" a passage
- or "translate" text into French

Respond with ONLY one word: 'summarize' or 'translate'.

User Input: \"{state['user_input']}\"

Answer:
"""
    response = llm.invoke(routing_prompt).strip().lower()
    
    # Extract task type from response
    if "summarize" in response:
        task_type = "summarize"
    elif "translate" in response:
        task_type = "translate"
    else:
        task_type = "summarize"  # Default
    
    return {**state, "task_type": task_type}

logger.info("✓ router_node defined")

In [ ]:
# Router function: Returns the task type for conditional routing
def router(state: RouterState) -> Literal["summarize", "translate"]:
    """Determines which node to route to based on task_type."""
    return state['task_type']

logger.info("✓ router function defined")

In [ ]:
# Handler nodes
def summarize_node(state: RouterState) -> RouterState:
    """Summarizes the user input."""
    prompt = f"Please provide a concise summary of the following passage:\n\n{state['user_input']}"
    response = llm.invoke(prompt)
    return {**state, "task_type": "summarize", "output": response}

def translate_node(state: RouterState) -> RouterState:
    """Translates the user input to French."""
    prompt = f"Translate the following text to French:\n\n{state['user_input']}"
    response = llm.invoke(prompt)
    return {**state, "task_type": "translate", "output": response}

logger.info("✓ Handler nodes defined (summarize_node, translate_node)")

In [ ]:
# Build the Routing workflow
workflow = StateGraph(RouterState)

# Add nodes
workflow.add_node("router", router_node)
workflow.add_node("summarize", summarize_node)
workflow.add_node("translate", translate_node)

# Set entry point
workflow.set_entry_point("router")

# Add conditional edges
workflow.add_conditional_edges("router", router, {
    "summarize": "summarize",
    "translate": "translate"
})

# Set finish points
workflow.set_finish_point("summarize")
workflow.set_finish_point("translate")

# Compile
app_routing = workflow.compile()

print_workflow_info(workflow)
logger.info("✓ Routing workflow compiled")

In [ ]:
# Test Routing - Example 1: Translation
print("\n" + "="*70)
print("ROUTING - TEST 1: TRANSLATION")
print("="*70)

input_text = {
    "user_input": "Can you translate this sentence: I love programming?",
    "task_type": "",
    "output": ""
}

logger.info("Executing Routing workflow (translation)...")
result = app_routing.invoke(input_text)

print(f"\nTask Type: {result['task_type']}")
print(f"Output:\n{result['output']}")

In [ ]:
# Test Routing - Example 2: Summarization
print("\n" + "="*70)
print("ROUTING - TEST 2: SUMMARIZATION")
print("="*70)

input_text = {
    "user_input": "Can you summarize this: I love programming so much it is the best thing ever. All I want to do is programming because it helps me solve problems and create innovative solutions.",
    "task_type": "",
    "output": ""
}

logger.info("Executing Routing workflow (summarization)...")
result = app_routing.invoke(input_text)

print(f"\nTask Type: {result['task_type']}")
print(f"Output:\n{result['output']}")

---
# PATRÓN 3: PARALLELIZATION

## Descripción

Parallelization es un patrón donde **múltiples tareas LLM se ejecutan al mismo tiempo** en lugar de una tras otra. Útil cuando diferentes partes de la tarea pueden hacerse **independientemente**, permitiendo procesamiento más rápido y mejor throughput del sistema.

**Casos de uso:**
- Resumir diferentes secciones de un documento grande simultáneamente
- Traducir un batch de mensajes de usuario a la vez
- Generar múltiples variaciones de ad copy o descripción de producto
- Ejecutar checks de seguridad usando diferentes prompts y comparar outputs
- Ensembling de resultados de diferentes modelos o prompts para consenso

## Use Case: Multilingual Translation Assistant

Construiremos un **asistente de traducción multilingüe** usando Parallelization. Tomaremos una sola oración en inglés y generaremos sus traducciones a francés, español y japonés — **todos al mismo tiempo**.

In [ ]:
import operator
from typing import Annotated

# Define state for Parallelization with reducer
class ParallelState(TypedDict):
    text: str
    french: str
    spanish: str
    japanese: str
    combined_output: str

logger.info("✓ ParallelState defined")

In [ ]:
# Translation nodes (to be executed in parallel)
def translate_french(state: ParallelState) -> dict:
    """Translates to French."""
    response = llm.invoke(f"Translate the following text to French:\n\n{state['text']}")
    return {"french": response.strip()}

def translate_spanish(state: ParallelState) -> dict:
    """Translates to Spanish."""
    response = llm.invoke(f"Translate the following text to Spanish:\n\n{state['text']}")
    return {"spanish": response.strip()}

def translate_japanese(state: ParallelState) -> dict:
    """Translates to Japanese."""
    response = llm.invoke(f"Translate the following text to Japanese:\n\n{state['text']}")
    return {"japanese": response.strip()}

logger.info("✓ Translation nodes defined")

In [ ]:
# Aggregator node: Combines all translation outputs
def aggregator(state: ParallelState) -> dict:
    """Combines all translations into a single output."""
    combined = f"Original Text: {state['text']}\n\n"
    combined += f"French: {state['french']}\n\n"
    combined += f"Spanish: {state['spanish']}\n\n"
    combined += f"Japanese: {state['japanese']}\n"
    return {"combined_output": combined}

logger.info("✓ aggregator node defined")

In [ ]:
# Build the Parallelization workflow
graph = StateGraph(ParallelState)

# Add nodes
graph.add_node("translate_french", translate_french)
graph.add_node("translate_spanish", translate_spanish)
graph.add_node("translate_japanese", translate_japanese)
graph.add_node("aggregator", aggregator)

# Connect parallel nodes from START
graph.add_edge(START, "translate_french")
graph.add_edge(START, "translate_spanish")
graph.add_edge(START, "translate_japanese")

# Connect all translation nodes to aggregator
graph.add_edge("translate_french", "aggregator")
graph.add_edge("translate_spanish", "aggregator")
graph.add_edge("translate_japanese", "aggregator")

# Connect aggregator to END
graph.add_edge("aggregator", END)

# Compile
app_parallel = graph.compile()

logger.info("✓ Parallelization workflow compiled")

In [ ]:
# Test Parallelization
print("\n" + "="*70)
print("PARALLELIZATION - MULTILINGUAL TRANSLATION")
print("="*70)

input_text = {
    "text": "Good morning! I hope you have a wonderful day.",
    "french": "",
    "spanish": "",
    "japanese": "",
    "combined_output": ""
}

logger.info("Executing Parallelization workflow...")
result = app_parallel.invoke(input_text)

print("\n" + result['combined_output'])

---
# Conclusión

Has completado exitosamente los tres patrones fundamentales de workflow agénticos con **Ollama Local**:

## 1. **Prompt Chaining** ⛓️
- Descompone tareas complejas en pasos secuenciales
- Cada paso alimenta el siguiente
- Ideal para pipelines dependientes

## 2. **Routing** 🚦
- Clasifica input y lo enruta a handlers especializados
- Soporta múltiples tipos de tarea
- Usa conditional edges en LangGraph

## 3. **Parallelization** ⚡
- Ejecuta múltiples tareas independientes al mismo tiempo
- Reduce latencia total
- Combina resultados con un nodo agregador

## Próximos Pasos

1. **Experimenta con los prompts** — modifica las instrucciones de cada nodo
2. **Agrega más handlers** — en el patrón Routing, agrega más opciones
3. **Combina patrones** — usa Routing + Chaining, o Chaining + Parallelization
4. **Integra tools** — agrega acceso a funciones externas vía tool calling
5. **Escala a producción** — añade logging, monitoring, y error handling robusto

## ✅ 100% Local, 100% Offline

Todos los LLM calls corre localmente en Ollama. **Sin dependencias de APIs cloud.** Soberanía de datos garantizada.